# Does AdaBoost get worse when we add more trees

[Master Tree Ensemble Models course](https://www.trainindata.com/p/master-tree-ensemble-models)

While preparing the boosting section of the course, we found that an AdaBoost that predicts the rent of an apartment becomes worse as we add trees to the ensemble, and this is the opposite of what we expect from an ensemble model.

In this notebook we find out whether this also happens in classification, we repeat the experiment in several datasets, we show that the shape of the target is the reason, and we see that a small learning rate keeps the problem under control.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import (
    fetch_california_housing,
    fetch_openml,
    load_diabetes,
    make_classification,
    make_regression,
)
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.metrics import roc_auc_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore")

### The experiment

AdaBoost adds the trees one after the other, so with `staged_predict` we can evaluate the ensemble after each new tree, and we obtain the whole curve by training the model only once.

We write two functions, one for classification and one for regression, that return the performance in the test set after every tree.

In [ ]:
def prepare(X, y, max_rows=20000):
    X = pd.DataFrame(X).copy()
    for var in X.columns:
        if str(X[var].dtype) in ("object", "category", "str"):
            X[var] = X[var].astype("category").cat.codes
    # AdaBoost does not take missing values, so we replace them by a value outside the distribution
    X = X.astype(float).fillna(-1)
    y = pd.Series(np.asarray(y))
    # we work with a sample of the largest datasets, to keep the notebook fast
    if len(X) > max_rows:
        X, _, y, _ = train_test_split(X, y, train_size=max_rows, random_state=0)
    return X, y

In [ ]:
def classification_curve(X, y, n_estimators=500, max_depth=1):
    X, y = prepare(X, y)
    y = y.astype("category").cat.codes
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=0
    )
    model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=max_depth),
        n_estimators=n_estimators,
        random_state=0,
    )
    model.fit(X_train, y_train)
    return pd.Series(
        [roc_auc_score(y_test, p[:, 1]) for p in model.staged_predict_proba(X_test)],
        index=range(1, len(model.estimators_) + 1),
    )


def regression_curve(X, y, n_estimators=100, max_depth=3, learning_rate=1.0):
    X, y = prepare(X, y)
    y = y.astype(float)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
    model = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=max_depth),
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=0,
    )
    model.fit(X_train, y_train)
    return pd.Series(
        [root_mean_squared_error(y_test, p) for p in model.staged_predict(X_test)],
        index=range(1, len(model.estimators_) + 1),
    )

In [ ]:
def summarise(curves, higher_is_better):
    summary = {}
    for name, curve in curves.items():
        best = curve.max() if higher_is_better else curve.min()
        summary[name] = {
            "best": best,
            "best at tree": curve.idxmax() if higher_is_better else curve.idxmin(),
            "final": curve.iloc[-1],
            "lost from best %": 100 * abs(curve.iloc[-1] - best) / best,
        }
    return pd.DataFrame(summary).T.round(4)

### Classification

We start with data that we make ourselves, so that we control how hard the problem is. We train one model with clean labels, and then we flip the labels of ten and twenty percent of the observations, because the weakness that everybody mentions about AdaBoost is that it chases the observations that it predicts badly, and a wrong label is an observation that no model can ever predict well.

We train 500 trees, which is far more than anybody would use in practice.

In [ ]:
X, y = make_classification(n_samples=6000, n_features=20, n_informative=6, random_state=0)

curves = {"clean labels": classification_curve(X, y)}
for noise in [0.1, 0.2]:
    rng = np.random.default_rng(0)
    noisy = np.where(rng.random(len(y)) < noise, 1 - y, y)
    curves[f"{int(noise * 100)}% wrong labels"] = classification_curve(X, noisy)

synthetic_curves = pd.DataFrame(curves)
summarise(curves, higher_is_better=True)

In [ ]:
synthetic_curves.plot()
plt.xlabel("Number of trees")
plt.ylabel("roc-auc in the test set")
plt.title("AdaBoost with data that we made ourselves")
plt.show()

The labels with noise bring the curves down, which is expected, and yet none of the curves turns downwards as we add trees. Let us now try with real data.

In [ ]:
datasets = {
    "credit-g": 31,
    "diabetes": 37,
    "spambase": 44,
    "phoneme": 1489,
    "churn": 40701,
}

curves = {}
for name, data_id in datasets.items():
    data = fetch_openml(data_id=data_id, as_frame=True)
    curves[name] = classification_curve(data.data, data.target)

classification_curves = pd.DataFrame(curves)
summarise(curves, higher_is_better=True)

In [ ]:
classification_curves.plot()
plt.xlabel("Number of trees")
plt.ylabel("roc-auc in the test set")
plt.title("AdaBoost in five classification datasets")
plt.legend(loc="lower right")
plt.show()

The performance improves quickly with the first trees and then it flattens, and even with 500 trees the models that lose something from their best point lose less than two percent of it. In classification we can therefore add trees without fear.

### Regression

Now we repeat the experiment with six datasets, the rent of the apartments that we use in the course and five datasets that we download.

In [ ]:
rent = pd.read_csv("../prepare-data/apartment_rent.csv", index_col="time")
california = fetch_california_housing(as_frame=True)
house_sales = fetch_openml(data_id=42731, as_frame=True)
ames = fetch_openml(data_id=42165, as_frame=True)
abalone = fetch_openml(data_id=44956, as_frame=True)
diabetes = load_diabetes(as_frame=True)

datasets = {
    "apartment rent": (rent.drop(columns="price"), rent["price"]),
    "house sales": (house_sales.data, house_sales.target),
    "california housing": (california.data, california.target),
    "ames housing": (ames.data, ames.target),
    "abalone": (abalone.data, abalone.target),
    "diabetes": (diabetes.data, diabetes.target),
}

curves = {name: regression_curve(X, y) for name, (X, y) in datasets.items()}

summary = summarise(curves, higher_is_better=False)
summary["skew of the target"] = [
    pd.Series(np.asarray(y, dtype=float)).skew() for _, y in datasets.values()
]
summary.sort_values("lost from best %", ascending=False).round(2)

In [ ]:
# the targets are in different units, so we divide each curve by its own best value
relative = pd.DataFrame({name: curve / curve.min() for name, curve in curves.items()})

relative.plot()
plt.xlabel("Number of trees")
plt.ylabel("rmse divided by the best rmse")
plt.title("AdaBoost in six regression datasets")
plt.show()

The datasets with a long right tail, the house sales, the rent and the houses of California, reach their best point with less than ten trees and then they climb, while the datasets with a tame target stay flat.

The relationship is a tendency and not a law. The houses of Ames have a skew of almost two and they behave perfectly well, so the shape of the target tells us when to be careful, and it does not tell us exactly how much we will suffer.

### Is the target really the reason

To answer this we keep the features fixed and we change only the target. We make a target that follows a normal distribution, then we make it log normal, then we make it even more extreme, then we take a normal target and we move one percent of the observations far away from the rest, and finally we take the logarithm of the log normal target, which should bring us back to where we started.

In [ ]:
X, y = make_regression(n_samples=6000, n_features=20, n_informative=6, noise=10.0, random_state=0)
z = (y - y.mean()) / y.std()

with_outliers = y.copy()
extreme = np.random.default_rng(0).choice(len(y), 60, replace=False)
with_outliers[extreme] = with_outliers[extreme] + 20 * y.std()

targets = {
    "normal": y,
    "log normal": np.exp(z),
    "very skewed": np.exp(2 * z),
    "normal with 1% extreme values": with_outliers,
    "logarithm of the log normal": z,
}

curves = {name: regression_curve(X, target) for name, target in targets.items()}

summary = summarise(curves, higher_is_better=False)
summary["skew of the target"] = [pd.Series(target).skew() for target in targets.values()]
summary.round(2)

With a normal target the curve never turns, with a skewed target the model falls apart, and the logarithm of that same target brings the good behaviour back. The fourth row is the clearest one, because a perfectly normal target with sixty extreme observations, which are one percent of the data, is enough to make the model several times worse than its best point.

The reason is in the algorithm. AdaBoost divides the error of every observation by the largest error of the round, so when one observation is far away from the rest, the errors of everybody else look like zero, the weights concentrate on a handful of observations, and the model follows them.

### Does a small learning rate help

The learning rate decides how much each tree moves the weights, so with a small learning rate the weights should drift slowly. We train the same four datasets with four learning rates, and we give the models 200 trees so that the slow ones have time to arrive.

In [ ]:
cases = {
    "synthetic log normal": (X, np.exp(z)),
    "apartment rent": (rent.drop(columns="price"), rent["price"]),
    "california housing": (california.data, california.target),
    "house sales": (house_sales.data, house_sales.target),
}

results = {}
for name, (features, target) in cases.items():
    for learning_rate in [1.0, 0.5, 0.1, 0.05]:
        curve = regression_curve(features, target, n_estimators=200, learning_rate=learning_rate)
        results[(name, learning_rate)] = {
            "best rmse": curve.min(),
            "best at tree": curve.idxmin(),
            "lost from best %": 100 * (curve.iloc[-1] - curve.min()) / curve.min(),
        }

results = pd.DataFrame(results).T
results.index.names = ["dataset", "learning rate"]
results.round(2)

In [ ]:
damage = results["lost from best %"].unstack()

damage.T.plot(marker="o")
plt.xlabel("Learning rate")
plt.ylabel("rmse lost from the best point, in %")
plt.title("A small learning rate keeps the deterioration under control")
plt.show()

With a learning rate of one the models lose between twenty and one hundred and thirteen percent of their best performance, and with a learning rate of 0.05 they lose almost nothing.

The best rmse is nearly the same in every row of the table, so the slow learning rate does not give us a worse model that deteriorates less. It gives us the same model, and it keeps it there.

### What we take home

In classification we can add trees to an AdaBoost and the performance flattens, even with five hundred trees and with labels that are wrong, so the number of trees is a parameter that we can set generously.

In regression the number of trees is a real hyperparameter that we need to tune, because AdaBoost measures the error of each observation relative to the largest error of the round, and a target with a long tail sends the weights to a handful of extreme observations.

When the target has a long tail we have two cheap solutions, the logarithm of the target and a small learning rate, and we can use both at the same time.